# Gradient Boosting Machines
Companion to the gradient boosting notes.

## 1. Regression boosting by hand: fit stumps to residuals

In [ ]:
import numpy as np
x = np.array([1.0, 2.0, 4.0, 5.0]); y = np.array([2.0, 5.0, 11.0, 14.0])
eta = 0.5
F = np.full(4, y.mean())
print("F0:", F, " MSE:", np.mean((y - F) ** 2))
for m in range(1, 4):
    r = y - F                                            # residuals = pseudo-residuals for squared error
    best = None
    for t in (x[:-1] + x[1:]) / 2:                       # every stump split
        left = x <= t
        pred = np.where(left, r[left].mean(), r[~left].mean())
        sse = ((r - pred) ** 2).sum()
        if best is None or sse < best[0]: best = (sse, t, pred)
    F = F + eta * best[2]
    print(f"tree {m}: split x <= {best[1]}, F = {F.round(4)}, MSE = {np.mean((y - F) ** 2):.4f}")

## 2. Classification boosting by hand: Newton leaf values

In [ ]:
sigmoid = lambda z: 1 / (1 + np.exp(-z))
x = np.array([1.2, 2.1, 3.8, 5.0]); y = np.array([0, 0, 1, 1])
F = np.zeros(4)
for m in range(1, 4):
    p = sigmoid(F); r = y - p
    left = x <= 3.0
    gamma = np.where(left, r[left].sum() / (p[left] * (1 - p[left])).sum(),
                           r[~left].sum() / (p[~left] * (1 - p[~left])).sum())
    F = F + 0.5 * gamma
    p = sigmoid(F)
    print(f"tree {m}: leaf values {np.unique(gamma).round(4)}, p = {p.round(4)}, log-loss = {-np.mean(y*np.log(p) + (1-y)*np.log(1-p)):.4f}")

## 3. The bank data

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, log_loss

bd = pd.read_csv("../../data/bd_train.csv").drop(columns=["REF_NO", "post_code", "post_area"])
y = (bd.pop("Revenue.Grid") == 1).astype(int)
X = pd.get_dummies(bd, drop_first=True, dtype=float)
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=1, stratify=y)
# hold out part of the training data to choose the number of trees (never the test set)
x_fit, x_val, y_fit, y_val = train_test_split(x_train, y_train, test_size=0.2, random_state=0, stratify=y_train)

## 4. Learning rate vs number of trees (validation data, never the test set)

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

for lr in [1.0, 0.3, 0.1, 0.03]:
    gbm = GradientBoostingClassifier(n_estimators=1000, learning_rate=lr, max_depth=3, random_state=0)
    gbm.fit(x_fit, y_fit)
    val_loss = [log_loss(y_val, p[:, 1]) for p in gbm.staged_predict_proba(x_val)]
    best = int(np.argmin(val_loss)) + 1
    print(f"learning_rate={lr:<5}: best number of trees {best:4d}  validation log-loss {min(val_loss):.4f}"
          f"  (after 1000 trees: {val_loss[-1]:.4f})")

## 5. Tree depth

In [ ]:
for depth in [1, 3, 6, 10]:
    gbm = GradientBoostingClassifier(n_estimators=1000, learning_rate=0.1, max_depth=depth, random_state=0).fit(x_fit, y_fit)
    val_loss = [log_loss(y_val, p[:, 1]) for p in gbm.staged_predict_proba(x_val)]
    print(f"max_depth={depth:2d}: best trees {int(np.argmin(val_loss)) + 1:4d}  best validation log-loss {min(val_loss):.4f}")

## 6. Final model with early stopping

In [ ]:
final = GradientBoostingClassifier(n_estimators=2000, learning_rate=0.1, max_depth=3, subsample=0.8,
                                   n_iter_no_change=30, validation_fraction=0.2, random_state=0)
final.fit(x_train, y_train)            # early stopping uses its own internal validation split
print("trees actually built:", final.n_estimators_)
print("test AUC:", round(roc_auc_score(y_test, final.predict_proba(x_test)[:, 1]), 4))

## 7. Regression on the loan data

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error

ld = pd.read_csv("../../data/loan_data_train.csv")
ld["Interest.Rate"] = pd.to_numeric(ld["Interest.Rate"].str.replace("%", ""), errors="coerce")
ld["FICO"] = ld["FICO.Range"].str.split("-", expand=True).astype(float).mean(axis=1)
ld["Loan.Length"] = pd.to_numeric(ld["Loan.Length"].str.replace(" months", ""), errors="coerce")
for col in ["Amount.Requested", "Open.CREDIT.Lines", "Revolving.CREDIT.Balance"]:
    ld[col] = pd.to_numeric(ld[col], errors="coerce")
cols = ["FICO", "Loan.Length", "Amount.Requested", "Inquiries.in.the.Last.6.Months",
        "Monthly.Income", "Open.CREDIT.Lines", "Revolving.CREDIT.Balance"]
ld = ld[cols + ["Interest.Rate"]].dropna()
xr_train, xr_test, yr_train, yr_test = train_test_split(ld[cols], ld["Interest.Rate"], test_size=0.2, random_state=2)
reg = GradientBoostingRegressor(n_estimators=2000, learning_rate=0.05, max_depth=3, subsample=0.8,
                                n_iter_no_change=30, validation_fraction=0.2, random_state=0).fit(xr_train, yr_train)
print("trees:", reg.n_estimators_, " test MAE:", round(mean_absolute_error(yr_test, reg.predict(xr_test)), 3))

## Try it yourself
1. Pseudo-residual for absolute error |y - F|: change section 1 to fit np.sign(r) and compare.
2. `subsample=0.5` in section 4 with eta 0.1.
3. Why must the number of trees not be chosen on the test set?